# 01 - Entendimiento inicial de datos

## Objetivo del notebook

Este notebook tiene como objetivo realizar el entendimiento inicial del dataset **Brazilian E-Commerce Public Dataset by Olist**. En esta fase se busca:

- Verificar que los 9 archivos CSV estén correctamente ubicados en `data/raw/`
- Conocer la estructura de cada tabla (dimensiones, columnas, tipos de datos)
- Identificar valores nulos y duplicados
- Revisar la cobertura temporal de los datos
- Validar las llaves primarias y relaciones esperadas entre tablas

In [1]:
# Importación de librerías
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [2]:
# Configuración de rutas
# Definir directorio raíz del proyecto (funciona desde notebooks/ o desde raíz)
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_RAW_DIR = ROOT_DIR / 'data' / 'raw'

print(f"Directorio raíz: {ROOT_DIR}")
print(f"Directorio de datos crudos: {DATA_RAW_DIR}")
print(f"¿Existe el directorio data/raw? {DATA_RAW_DIR.exists()}")

Directorio raíz: c:\Users\Samuel Vergara\Desktop\Apps\marketpulse-olist-commerce-intelligence
Directorio de datos crudos: c:\Users\Samuel Vergara\Desktop\Apps\marketpulse-olist-commerce-intelligence\data\raw
¿Existe el directorio data/raw? True


In [3]:
# Verificación de archivos esperados
# Lista de 9 archivos esperados del dataset Olist
archivos_esperados = [
    'olist_orders_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_order_payments_dataset.csv',
    'olist_order_reviews_dataset.csv',
    'olist_customers_dataset.csv',
    'olist_products_dataset.csv',
    'olist_sellers_dataset.csv',
    'olist_geolocation_dataset.csv',
    'product_category_name_translation.csv'
]

# Verificar cuáles existen
verificacion_archivos = []
for archivo in archivos_esperados:
    ruta_completa = DATA_RAW_DIR / archivo
    existe = ruta_completa.exists()
    verificacion_archivos.append({
        'archivo': archivo,
        'existe': existe,
        'ruta': str(ruta_completa) if existe else 'No encontrado'
    })

# Crear DataFrame con resultados
df_verificacion = pd.DataFrame(verificacion_archivos)
display(df_verificacion)

# Mostrar advertencia si faltan archivos
archivos_faltantes = df_verificacion[~df_verificacion['existe']]
if len(archivos_faltantes) > 0:
    print(f"\n⚠️ ADVERTENCIA: Faltan {len(archivos_faltantes)} archivo(s):")
    for idx, row in archivos_faltantes.iterrows():
        print(f"  - {row['archivo']}")
else:
    print("\n✓ Todos los archivos esperados están presentes.")

,archivo,existe,ruta
0,olist_orders_dataset.csv,True,c:\Users\Samuel Vergara\Desktop\Apps\marketpul...
1,olist_order_items_dataset.csv,True,c:\Users\Samuel Vergara\Desktop\Apps\marketpul...
2,olist_order_payments_dataset.csv,True,c:\Users\Samuel Vergara\Desktop\Apps\marketpul...
3,olist_order_reviews_dataset.csv,True,c:\Users\Samuel Vergara\Desktop\Apps\marketpul...
4,olist_customers_dataset.csv,True,c:\Users\Samuel Vergara\Desktop\Apps\marketpul...
5,olist_products_dataset.csv,True,c:\Users\Samuel Vergara\Desktop\Apps\marketpul...
6,olist_sellers_dataset.csv,True,c:\Users\Samuel Vergara\Desktop\Apps\marketpul...
7,olist_geolocation_dataset.csv,True,c:\Users\Samuel Vergara\Desktop\Apps\marketpul...
8,product_category_name_translation.csv,True,c:\Users\Samuel Vergara\Desktop\Apps\marketpul...



✓ Todos los archivos esperados están presentes.


In [4]:
# Carga de datos
# Cargar cada CSV existente en un diccionario de dataframes
dataframes = {}

# Mapeo de nombres de archivos a nombres internos simples
mapeo_archivos = {
    'olist_orders_dataset.csv': 'orders',
    'olist_order_items_dataset.csv': 'order_items',
    'olist_order_payments_dataset.csv': 'payments',
    'olist_order_reviews_dataset.csv': 'reviews',
    'olist_customers_dataset.csv': 'customers',
    'olist_products_dataset.csv': 'products',
    'olist_sellers_dataset.csv': 'sellers',
    'olist_geolocation_dataset.csv': 'geolocation',
    'product_category_name_translation.csv': 'category_translation'
}

# Cargar solo los archivos que existen
for archivo, nombre_interno in mapeo_archivos.items():
    ruta_completa = DATA_RAW_DIR / archivo
    if ruta_completa.exists():
        try:
            dataframes[nombre_interno] = pd.read_csv(ruta_completa)
            print(f"✓ Cargado: {archivo} -> {nombre_interno}")
        except Exception as e:
            print(f"✗ Error al cargar {archivo}: {e}")
    else:
        print(f"⊘ No encontrado: {archivo}")

print(f"\nTotal de dataframes cargados: {len(dataframes)}")

✓ Cargado: olist_orders_dataset.csv -> orders
✓ Cargado: olist_order_items_dataset.csv -> order_items
✓ Cargado: olist_order_payments_dataset.csv -> payments
✓ Cargado: olist_order_reviews_dataset.csv -> reviews
✓ Cargado: olist_customers_dataset.csv -> customers
✓ Cargado: olist_products_dataset.csv -> products
✓ Cargado: olist_sellers_dataset.csv -> sellers
✓ Cargado: olist_geolocation_dataset.csv -> geolocation
✓ Cargado: product_category_name_translation.csv -> category_translation

Total de dataframes cargados: 9


In [5]:
# Resumen general de tablas
# Crear tabla resumen con dimensiones, nulos y duplicados
resumen_tablas = []

for nombre, df in dataframes.items():
    filas = df.shape[0]
    columnas = df.shape[1]
    columnas_con_nulos = df.isnull().any().sum()
    total_nulos = df.isnull().sum().sum()
    filas_duplicadas = df.duplicated().sum()
    
    resumen_tablas.append({
        'nombre_tabla': nombre,
        'filas': filas,
        'columnas': columnas,
        'columnas_con_nulos': columnas_con_nulos,
        'total_nulos': total_nulos,
        'filas_duplicadas': filas_duplicadas
    })

df_resumen = pd.DataFrame(resumen_tablas)
df_resumen = df_resumen.sort_values('filas', ascending=False).reset_index(drop=True)
display(df_resumen)

,nombre_tabla,filas,columnas,columnas_con_nulos,total_nulos,filas_duplicadas
0,geolocation,1000163,5,0,0,261831
1,order_items,112650,7,0,0,0
2,payments,103886,5,0,0,0
3,orders,99441,8,3,4908,0
4,customers,99441,5,0,0,0
5,reviews,99224,7,2,145903,0
6,products,32951,9,8,2448,0
7,sellers,3095,4,0,0,0
8,category_translation,71,2,0,0,0


In [6]:
# Vista inicial de cada tabla
# Mostrar información básica de cada dataframe
for nombre, df in dataframes.items():
    print(f"\n{'='*60}")
    print(f"Tabla: {nombre}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape[0]} filas × {df.shape[1]} columnas")
    print(f"\nPrimeras 5 filas:")
    display(df.head())
    print(f"\nTipos de datos:")
    print(df.dtypes)


Tabla: orders
Shape: 99441 filas × 8 columnas

Primeras 5 filas:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00



Tipos de datos:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

Tabla: order_items
Shape: 112650 filas × 7 columnas

Primeras 5 filas:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14



Tipos de datos:
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

Tabla: payments
Shape: 103886 filas × 5 columnas

Primeras 5 filas:


...


Tipos de datos:
seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object

Tabla: geolocation
Shape: 1000163 filas × 5 columnas

Primeras 5 filas:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP



Tipos de datos:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

Tabla: category_translation
Shape: 71 filas × 2 columnas

Primeras 5 filas:


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor



Tipos de datos:
product_category_name            str
product_category_name_english    str
dtype: object


In [7]:
# Revisión de columnas por tabla
# Crear tabla detallada con información de cada columna
revision_columnas = []

for nombre, df in dataframes.items():
    for columna in df.columns:
        tipo_dato = str(df[columna].dtype)
        nulos = df[columna].isnull().sum()
        porcentaje_nulos = (nulos / len(df)) * 100
        valores_unicos = df[columna].nunique()
        
        revision_columnas.append({
            'nombre_tabla': nombre,
            'columna': columna,
            'tipo_dato': tipo_dato,
            'nulos': nulos,
            'porcentaje_nulos': round(porcentaje_nulos, 2),
            'valores_unicos': valores_unicos
        })

df_revision_columnas = pd.DataFrame(revision_columnas)
display(df_revision_columnas)

,nombre_tabla,columna,tipo_dato,nulos,porcentaje_nulos,valores_unicos
0,orders,order_id,str,0,0.00,99441
1,orders,customer_id,str,0,0.00,99441
2,orders,order_status,str,0,0.00,8
3,orders,order_purchase_timestamp,str,0,0.00,98875
4,orders,order_approved_at,str,160,0.16,90733
5,orders,order_delivered_carrier_date,str,1783,1.79,81018
6,orders,order_delivered_customer_date,str,2965,2.98,95664
7,orders,order_estimated_delivery_date,str,0,0.00,459
8,order_items,order_id,str,0,0.00,98666
9,order_items,order_item_id,int64,0,0.00,21


In [8]:
# Revisión básica de fechas
# Para la tabla orders, revisar columnas de fecha si existen
if 'orders' in dataframes:
    df_orders = dataframes['orders']
    
    columnas_fecha = [
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
    
    columnas_fecha_existentes = [col for col in columnas_fecha if col in df_orders.columns]
    
    if columnas_fecha_existentes:
        print("Análisis de columnas de fecha en tabla orders:")
        print("="*60)
        
        revision_fechas = []
        for col in columnas_fecha_existentes:
            # Convertir temporalmente a datetime para diagnóstico
            temp_series = pd.to_datetime(df_orders[col], errors='coerce')
            
            fecha_min = temp_series.min()
            fecha_max = temp_series.max()
            nulos = df_orders[col].isnull().sum()
            
            revision_fechas.append({
                'columna': col,
                'fecha_minima': fecha_min,
                'fecha_maxima': fecha_max,
                'nulos': nulos
            })
            
            print(f"\n{col}:")
            print(f"  - Fecha mínima: {fecha_min}")
            print(f"  - Fecha máxima: {fecha_max}")
            print(f"  - Nulos: {nulos}")
        
        df_revision_fechas = pd.DataFrame(revision_fechas)
        display(df_revision_fechas)
    else:
        print("No se encontraron columnas de fecha en la tabla orders.")
else:
    print("La tabla orders no está disponible.")

Análisis de columnas de fecha en tabla orders:

order_purchase_timestamp:
  - Fecha mínima: 2016-09-04 21:15:19
  - Fecha máxima: 2018-10-17 17:30:18
  - Nulos: 0

order_approved_at:
  - Fecha mínima: 2016-09-15 12:16:38
  - Fecha máxima: 2018-09-03 17:40:06
  - Nulos: 160

order_delivered_carrier_date:
  - Fecha mínima: 2016-10-08 10:34:01
  - Fecha máxima: 2018-09-11 19:48:28
  - Nulos: 1783

order_delivered_customer_date:
  - Fecha mínima: 2016-10-11 13:46:32
  - Fecha máxima: 2018-10-17 13:22:46
  - Nulos: 2965

order_estimated_delivery_date:
  - Fecha mínima: 2016-09-30 00:00:00
  - Fecha máxima: 2018-11-12 00:00:00
  - Nulos: 0


,columna,fecha_minima,fecha_maxima,nulos
0,order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18,0
1,order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06,160
2,order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28,1783
3,order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46,2965
4,order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00,0


In [10]:
# Revisión básica de llaves
# Validar conteos únicos de columnas importantes
revision_llaves = []

# orders: order_id, customer_id
if 'orders' in dataframes:
    df = dataframes['orders']
    if 'order_id' in df.columns:
        revision_llaves.append({'tabla': 'orders', 'columna': 'order_id', 'unicos': df['order_id'].nunique()})
    if 'customer_id' in df.columns:
        revision_llaves.append({'tabla': 'orders', 'columna': 'customer_id', 'unicos': df['customer_id'].nunique()})

# customers: customer_id, customer_unique_id
if 'customers' in dataframes:
    df = dataframes['customers']
    if 'customer_id' in df.columns:
        revision_llaves.append({'tabla': 'customers', 'columna': 'customer_id', 'unicos': df['customer_id'].nunique()})
    if 'customer_unique_id' in df.columns:
        revision_llaves.append({'tabla': 'customers', 'columna': 'customer_unique_id', 'unicos': df['customer_unique_id'].nunique()})

# order_items: order_id, product_id, seller_id
if 'order_items' in dataframes:
    df = dataframes['order_items']
    if 'order_id' in df.columns:
        revision_llaves.append({'tabla': 'order_items', 'columna': 'order_id', 'unicos': df['order_id'].nunique()})
    if 'product_id' in df.columns:
        revision_llaves.append({'tabla': 'order_items', 'columna': 'product_id', 'unicos': df['product_id'].nunique()})
    if 'seller_id' in df.columns:
        revision_llaves.append({'tabla': 'order_items', 'columna': 'seller_id', 'unicos': df['seller_id'].nunique()})

# payments: order_id
if 'payments' in dataframes:
    df = dataframes['payments']
    if 'order_id' in df.columns:
        revision_llaves.append({'tabla': 'payments', 'columna': 'order_id', 'unicos': df['order_id'].nunique()})

# reviews: order_id, review_id
if 'reviews' in dataframes:
    df = dataframes['reviews']
    if 'order_id' in df.columns:
        revision_llaves.append({'tabla': 'reviews', 'columna': 'order_id', 'unicos': df['order_id'].nunique()})
    if 'review_id' in df.columns:
        revision_llaves.append({'tabla': 'reviews', 'columna': 'review_id', 'unicos': df['review_id'].nunique()})

# products: product_id
if 'products' in dataframes:
    df = dataframes['products']
    if 'product_id' in df.columns:
        revision_llaves.append({'tabla': 'products', 'columna': 'product_id', 'unicos': df['product_id'].nunique()})

# sellers: seller_id
if 'sellers' in dataframes:
    df = dataframes['sellers']
    if 'seller_id' in df.columns:
        revision_llaves.append({'tabla': 'sellers', 'columna': 'seller_id', 'unicos': df['seller_id'].nunique()})

df_revision_llaves = pd.DataFrame(revision_llaves)
display(df_revision_llaves)

,tabla,columna,unicos
0,orders,order_id,99441
1,orders,customer_id,99441
2,customers,customer_id,99441
3,customers,customer_unique_id,96096
4,order_items,order_id,98666
5,order_items,product_id,32951
6,order_items,seller_id,3095
7,payments,order_id,99440
8,reviews,order_id,98673
9,reviews,review_id,98410
